# 🪱 JORINOVA NEXUS — Blood parasites / microfilaria detector (fine-tuning)

Detects microfilariae (Wuchereria, Brugia, Loa) and blood flagellates on blood film.

Produces `microfilaria.pt` for the app's vision service; detected classes resolve to
their disease/finding via `backend/ai_services/blood_parasite_organisms.json`.

> **Fine-tuning, NOT from scratch** — starts from `yolov8s.pt` and adapts.
> **Data source: KAGGLE** (not Roboflow) — upload your `kaggle.json` in step 4.
> ⚠️ Public microfilaria *detection* data is scarce — set `DATASET` to a real slug
> (see the search URL in step 4). Until then the app reads microfilaria via Claude vision.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_microfilaria', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_microfilaria')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (KAGGLE — not Roboflow)
Upload your `kaggle.json` (kaggle.com → Settings → API → **Create New Token**), then set
`DATASET` to a microfilaria dataset slug. Search:
**https://www.kaggle.com/datasets?search=microfilaria** (or `filaria`, `blood parasite`).
The cell auto-detects a YOLO export (`data.yaml`) or converts Pascal-VOC (`.xml`) → YOLO.

In [ ]:
# ── 4. Microfilaria data via KAGGLE (not Roboflow) -> DATA_YAML ──
# Public microfilaria DETECTION data is scarce. This cell takes ANY Kaggle dataset and:
#   (a) uses its YOLO data.yaml if present, else (b) converts Pascal-VOC .xml -> YOLO.
# Search: https://www.kaggle.com/datasets?search=microfilaria  (or filaria / blood parasite)
import os, glob, shutil, subprocess, sys, yaml
import xml.etree.ElementTree as ET

# 1) Kaggle token (kaggle.com -> Settings -> API -> Create New Token -> kaggle.json)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json ...'); up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    name = 'kaggle.json' if os.path.exists('kaggle.json') else list(up)[0]
    shutil.move(name, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

# 2) SET your dataset slug (owner/dataset-name) — replace when you have a microfilaria set
DATASET = 'owner/microfilaria-dataset'            # <-- REPLACE ME (see search URL above)
RAW = '/content/data/mf_raw'
if os.path.exists(RAW): shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)
rc = subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-p', RAW, '--unzip']).returncode
assert rc == 0, 'Kaggle download failed — check the DATASET slug and that you accepted its rules on kaggle.com.'
print('top-level:', os.listdir(RAW)[:20])

# 3a) already YOLO? use its data.yaml
yamls = glob.glob(RAW + '/**/data.yaml', recursive=True)
if yamls:
    DATA_YAML = yamls[0]
    print('YOLO dataset -> DATA_YAML =', DATA_YAML, '| classes:', yaml.safe_load(open(DATA_YAML))['names'])
else:
    # 3b) Pascal-VOC (.xml) -> YOLO
    xmls = glob.glob(RAW + '/**/*.xml', recursive=True)
    assert xmls, ('No data.yaml and no VOC .xml under ' + RAW + ' — print os.listdir(RAW) and send it to me; '
                  'if it is class-folders use the classification pattern instead.')
    names = []
    for x in xmls:
        for o in ET.parse(x).getroot().findall('object'):
            n = (o.findtext('name') or 'microfilaria').strip().lower().replace(' ', '_')
            if n not in names: names.append(n)
    OUT = '/content/data/mf_yolo'
    for x in xmls:
        r = ET.parse(x).getroot(); fn = r.findtext('filename') or ''
        img = None
        for cand in [fn, os.path.splitext(fn)[0] + '.jpg', os.path.splitext(fn)[0] + '.png',
                     os.path.splitext(os.path.basename(x))[0] + '.jpg']:
            hit = glob.glob(RAW + '/**/' + os.path.basename(cand), recursive=True) if cand else []
            if hit: img = hit[0]; break
        if not img: continue
        sz = r.find('size'); W = float(sz.findtext('width')); H = float(sz.findtext('height'))
        split = 'val' if (abs(hash(os.path.basename(img))) % 7 == 0) else 'train'
        oi = os.path.join(OUT, 'images', split); ol = os.path.join(OUT, 'labels', split)
        os.makedirs(oi, exist_ok=True); os.makedirs(ol, exist_ok=True)
        shutil.copy(img, oi)
        lines = []
        for o in r.findall('object'):
            n = (o.findtext('name') or 'microfilaria').strip().lower().replace(' ', '_')
            b = o.find('bndbox'); x1=float(b.findtext('xmin')); y1=float(b.findtext('ymin'))
            x2=float(b.findtext('xmax')); y2=float(b.findtext('ymax'))
            lines.append('%d %.6f %.6f %.6f %.6f' % (names.index(n), (x1+x2)/2/W, (y1+y2)/2/H, (x2-x1)/W, (y2-y1)/H))
        open(os.path.join(ol, os.path.splitext(os.path.basename(img))[0] + '.txt'), 'w').write('\n'.join(lines))
    DATA_YAML = os.path.join(OUT, 'data.yaml')
    open(DATA_YAML, 'w').write('path: %s\ntrain: images/train\nval: images/val\nnc: %d\nnames: %s\n' % (OUT, len(names), names))
    print('VOC->YOLO -> DATA_YAML =', DATA_YAML, '| classes:', names)

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8s.pt'   # yolov8m.pt = more accurate/slower
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_microfilaria/runs'
try:
    os.makedirs(PROJECT, exist_ok=True)
except Exception:
    PROJECT = '/content/runs/detect'
model = YOLO(BASE_WEIGHTS)
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
            project=PROJECT, name='microfilaria', pretrained=True, patience=25, plots=True,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=180, fliplr=0.5, flipud=0.5,
            mosaic=1.0, translate=0.1, scale=0.5)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 6b. RESUME if a run disconnects (continue, no 4-vs-80 crash)

In [ ]:
# Run INSTEAD of step 6 to continue a disconnected run from its Drive checkpoint.
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_microfilaria/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_microfilaria/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = YOLO(ckpt)                       # architecture comes FROM the checkpoint (correct classes)
try:
    if ckpt in runs: model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception:
    assert 'DATA_YAML' in globals(), 'Run step 4 first so DATA_YAML is set.'
    model = YOLO(ckpt); model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
        project='/content/drive/MyDrive/nexus_microfilaria/runs', name='microfilaria_resume', pretrained=True, patience=25)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_microfilaria'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try:
    os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/microfilaria.pt'); print('Drive ->', DRIVE + '/microfilaria.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`microfilaria.pt`** and place it at **`backend/models/microfilaria/microfilaria.pt`**, commit, push.
2. The vision service auto-loads it for `image_type == "microfilaria"` (weights at `models/microfilaria/microfilaria.pt`).
3. Detected classes resolve to disease/finding via `backend/ai_services/blood_parasite_organisms.json`.
4. On Render, build with `INSTALL_ML=1` (>=2 GB) to run the detector; else Claude vision reads the field.